In [1]:
!pip install rapidfuzz openpyxl requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.8 MB/s eta 0:00:00


In [2]:
# Download lexicon file (xlsx)
import requests

url = "https://huggingface.co/datasets/NLPC-UOM/sinhala-sentiment-lexicon-generation/resolve/main/words.xlsx"
response = requests.get(url)

with open("words.xlsx", "wb") as f:
    f.write(response.content)

print("Downloaded words.xlsx")

Downloaded words.xlsx


In [3]:
# Load the lexicon
import pandas as pd

df = pd.read_excel("words.xlsx")
print(df.head(10))
print("Columns:", df.columns.tolist())

         word  score
0         අංක      0
1     අකටයුතු     -1
2  අකටයුතුකම්     -1
3   අකටයුතුයි     -1
4       අකණ්ඩ      1
5      අකමැති     -1
6    අකමැතියි     -1
7     අකමැත්ත     -1
8  අකමැත්තෙන්     -1
9        අංකය      0
Columns: ['word', 'score']


In [4]:
# Build the lexicon set
word_column = df.columns[0]
lexicon = set(df[word_column].dropna().astype(str).str.strip().tolist())

print(f"Loaded {len(lexicon)} words into lexicon")

Loaded 12399 words into lexicon


In [5]:
# Correction function
from rapidfuzz import process, fuzz

def find_closest(word, lexicon, score_cutoff=70):
    result = process.extractOne(
        word,
        lexicon,
        scorer=fuzz.ratio,
        score_cutoff=score_cutoff
    )
    return result[0] if result else word  # return original if no close match

def correct_text(noisy_text, lexicon, score_cutoff=70):
    words = noisy_text.split()
    corrected = []
    for word in words:
        if word in lexicon:
            corrected.append(word)
        else:
            corrected.append(find_closest(word, lexicon, score_cutoff))
    return " ".join(corrected)

In [7]:
# Test pipeline
noisy_text = "අයිබවන් ම ප්‍රබත්මට පුළුවනි ොබර සහය වන්න"   # put STT output here

corrected = correct_text(noisy_text, lexicon, score_cutoff=70)

print("Original :", noisy_text)
print("Corrected:", corrected)

Original : අයිබවන් ම ප්‍රබත්මට පුළුවනි ොබර සහය වන්න
Corrected: අයුරින් ම ප්‍රබලම පුළුවනි ොබර සහය වන්න
